In [5]:
%matplotlib widget

In [6]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from g4beam import *
from scan import *
from scipy.optimize import differential_evolution
from matplotlib.cm import viridis
import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import re
import sys
from pathlib import Path
from matplotlib import cm
import numpy as np
import pandas as pd
from tqdm import *
import pickle
import itertools
import os
from tabulate import tabulate
import tempfile
import glob
import json
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [7]:
# Make Sure g4bl is here
import os
os.environ["PATH"] += os.pathsep + "/home/incik/G4beamline-3.08/bin"
import shutil
print(shutil.which("g4bl"))

/home/incik/G4beamline-3.08/bin/g4bl


## Creating a Single Run with Manually Specifed Parameters

### What Daniel Fu's Wedge Outputs
Note that we will not be using this file as input as we have the wedge inside the g4bl file. "particles_before.txt" is the file input to the g4bl files.

In [13]:
# If particles_after isn't updated with Z = 0.
def convertZ(input_file, output_file):
    event_id_counter = 1
    with open(input_file, "r") as infile, open(output_file, "w") as outfile:
        for line in infile:
            # Skip header lines (those starting with #)
            if line.strip().startswith("#"):
                outfile.write(line)
                continue

            # Split the line into columns
            parts = line.strip().split()
            if len(parts) >= 12:
                parts[2] = "0"  # Set the 3rd column (z) to 0
                # Replace event ID (assuming it's the 9th column, zero-based index 8)
                # Adjust if your event ID is in a different column
                parts[8] = str(event_id_counter)
                event_id_counter += 1
                new_line = " ".join(parts)
                outfile.write(new_line + "\n")
            else:
                # Handle lines that don't match expected format
                outfile.write(line)
    print(f"Updated file saved as '{output_file}'")
    os.remove(input_file)
    return None
convertZ("particles_after.txt", "particles_afterupt.txt")

Updated file saved as 'particles_afterupt.txt'


In [16]:
# Load data POST WEDGE as a Dataframe so that you can use Daniel Fu's functions
filename = "out_1760039204_1614938.txt"

# Skip the first two header lines that start with '#'
with open(filename) as f:
    # Read until the line containing column names
    for line in f:
        if line.startswith("#x "):
            columns = line.strip().lstrip("#").split()
            break

# Now load the data into a DataFrame
df = pd.read_csv(filename, comment="#", sep='\s+', names=columns)

x_params, y_params, z_emit = calc_all_params(df) # _params are tuples of the form (emittance, beta, gamma, alpha, D, D') ALL IN m
D_dict = {r"$\beta$_x": x_params[1], r"$\gamma$_x": x_params[2], r"$\alpha$_x": x_params[3],  "D_x": x_params[4], "D'_x": x_params[5], r"$\beta$_y": y_params[1], r"$\gamma$_y": y_params[2], r"$\alpha$_y": y_params[3], "D_y": y_params[4], "D'_y": y_params[5]}
print(D_dict)
print(r"Epsilon_z: "+str(z_emit))

{'$\\beta$_x': np.float64(0.04352034430332993), '$\\gamma$_x': np.float64(255.79365628939377), '$\\alpha$_x': np.float64(-3.183116082131164), 'D_x': np.float64(0.015274752174883286), "D'_x": np.float64(-0.16276468456961618), '$\\beta$_y': np.float64(0.022433027940989454), '$\\gamma$_y': np.float64(73.66612099940427), '$\\alpha$_y': np.float64(-0.8078082388066774), 'D_y': np.float64(-0.0003564703042096456), "D'_y": np.float64(-0.01884340044268174)}
Epsilon_z: 7.5011508670475155


<>:13: SyntaxWarning: invalid escape sequence '\s'
<>:13: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_5746/2525864303.py:13: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(filename, comment="#", sep='\s+', names=columns)


In [17]:
# Replace with your beam file
filename = "out_1760039204_1614938.txt"

# Load data assuming BLTrackFile columns: x y z px py pz t particleID ...
# Adjust columns if your file is different
data = np.loadtxt(filename, comments="#")

# Extract px, py, pz
px = data[:, 3]
py = data[:, 4]
pz = data[:, 5]

# Compute average momentum components
px_ref = np.mean(px)
py_ref = np.mean(py)
pz_ref = np.mean(pz)

# Compute average total momentum magnitude
p_ref = np.sqrt(px_ref**2 + py_ref**2 + pz_ref**2)

print(f"Reference particle momentum components: px={px_ref:.6f}, py={py_ref:.6f}, pz={pz_ref:.6f}")
print(f"Reference particle total momentum: p = {p_ref:.6f}")

Reference particle momentum components: px=-0.204087, py=-0.052419, pz=88.961592
Reference particle total momentum: p = 88.961842


In [18]:
desired_p = np.array([-0.204087, -0.204087, 88.961842])  # px, py, pz in MeV/c

data = np.loadtxt("out_1760039204_1614938.txt")
momenta = data[:, 3:6]  # px, py, pz

# Find the particle index closest to desired momentum
dist = np.linalg.norm(momenta - desired_p, axis=1)
ref_index = np.argmin(dist)

print("Use this index in referenceParticle:", ref_index)


Use this index in referenceParticle: 4072


## Analytic Calculation of the Lattice from Wedge

1. Getting the Twiss Parameters of the Post-Wedge particle distribution

In [23]:
filename = "out_1760039204_1614938.txt" # end of wedge with optimal parameters

D_dict = {"epsilon_x": x_params[0], "beta_x": x_params[1], "gamma_x": x_params[2], "alpha_x": x_params[3],  "D_x": x_params[4], "D'_x": x_params[5], 
        "epsilon_y": y_params[0], "beta_y": y_params[1], "gamma_y": y_params[2], "alpha_y": y_params[3], "D_y": y_params[4], "D'_y": y_params[5],
        "epsilon_z": z_emit}
print(D_dict)

{'epsilon_x': np.float64(0.032314888033942994), 'beta_x': np.float64(0.04352034430332993), 'gamma_x': np.float64(255.79365628939377), 'alpha_x': np.float64(-3.183116082131164), 'D_x': np.float64(0.015274752174883286), "D'_x": np.float64(-0.16276468456961618), 'epsilon_y': np.float64(0.11683931480995156), 'beta_y': np.float64(0.022433027940989454), 'gamma_y': np.float64(73.66612099940427), 'alpha_y': np.float64(-0.8078082388066774), 'D_y': np.float64(-0.0003564703042096456), "D'_y": np.float64(-0.01884340044268174), 'epsilon_z': np.float64(7.5011508670475155)}


2. Build K(s) coming directly from the lattice created by magnets: the idea is that we can build the K(s) in the Hill's equation:
$u'' + K(s)u(s) = 0$. K(s) can be built piecewise:
\begin{equation}
    K(s)=0 for s ∈ [s0​,s0​+L) \text{  for Drift of length L}
\end{equation}

\begin{equation}
    K(s)_x = \frac{G}{B\rho},  
    K(s)_y = -\frac{G}{B\rho} \text{ for a Quadrupole of length L and gradient $G = \partial B/ \partial x$}
\end{equation}

\begin{equation}
    K​(s)_x = \frac{1}{\rho^2}​,
    K​(s)_y = 0 \text{ for a pure dipole}
\end{equation}

In [ ]:
elements = [
        {'type':'quad','L':0.5,'k':1.2},
        {'type':'quad','L':0.5,'k':-1.0},
        {'type':'drift','L':0.3,'k':0.0},
        {'type':'dip','L':1.0,'rho':20.0}  # dipole with bending radius -> nonzero h
    ]
    K_of_s, h_of_s, Ltot = make_piecewise_K(elements)

In [ ]:

# ---------------------------
# 2) Build K(s) piecewise
# elements = list of (type, length, k, rho)
# type: 'quad' (k = focusing strength, rho ignored),
#       'drift' (k=0),
#       'dip' (k=0, rho = bending radius -> h=1/rho nonzero)
# We'll create K(s) and h(s) (1/rho)
# ---------------------------
def make_piecewise_K(elements):
    # elements: list of dicts: {'type':'quad','L':..., 'k':...}, etc.
    seg_starts = [0.0]
    for e in elements:
        seg_starts.append(seg_starts[-1] + e['L'])
    total_L = seg_starts[-1]

    # arrays for segments
    segs = []
    for i,e in enumerate(elements):
        s0 = seg_starts[i]
        s1 = seg_starts[i+1]
        segs.append((s0, s1, e))

    def K_of_s(s):
        # piecewise constant
        # allow scalar or array
        s = np.asarray(s)
        out = np.zeros_like(s, dtype=float)
        for (s0,s1,e) in segs:
            mask = (s >= s0) & (s < s1)
            if e['type']=='quad':
                out[mask] = e.get('k',0.0)
            else:
                out[mask] = 0.0
        # last point included
        if np.isscalar(s):
            return float(out)
        return out

    def h_of_s(s):
        s = np.asarray(s)
        out = np.zeros_like(s, dtype=float)
        for (s0,s1,e) in segs:
            mask = (s >= s0) & (s < s1)
            if e['type']=='dip':
                rho = e.get('rho', np.inf)
                out[mask] = 1.0/rho if np.isfinite(rho) else 0.0
        if np.isscalar(s):
            return float(out)
        return out

    return K_of_s, h_of_s, total_L

# ---------------------------
# 3) Integrate B(s) (envelope)
# ---------------------------
def integrate_B(B0, Bp0, K_of_s, s_span, dense_output=False):
    # convert second order -> first order system for [B, B']
    def fun(s, y):
        B, Bp = y
        K = K_of_s(np.array([s])) if callable(K_of_s) else K_of_s(s)
        # B'' = (2/B)*(1 + 1/4 B'^2 - B^2 K)
        Bpp = (2.0/B)*(1.0 + 0.25*(Bp**2) - (B**2)*K)
        return [Bp, Bpp]

    sol = solve_ivp(fun, (0.0, s_span), [B0, Bp0], dense_output=dense_output, rtol=1e-8, atol=1e-10)
    return sol

# ---------------------------
# 4) Fundamental matrix (transfer) by integrating 4 eqns
# Y' = A(s) Y with A = [[0,1],[-K(s),0]]
# columns of Phi(s) are solutions starting from I
# ---------------------------
def transfer_matrix(K_of_s, s_end):
    # integrate the 4 components of Phi flattened column-major
    def fun(s, y):
        # y = [phi11, phi21, phi12, phi22] where phi = [[phi11,phi12],[phi21,phi22]]
        phi11, phi21, phi12, phi22 = y
        K = K_of_s(np.array([s])) if callable(K_of_s) else K_of_s(s)
        # derivatives according to A*phi
        dphi11 = phi21
        dphi21 = -K*phi11
        dphi12 = phi22
        dphi22 = -K*phi12
        return [dphi11, dphi21, dphi12, dphi22]

    y0 = [1.0, 0.0, 0.0, 1.0]
    sol = solve_ivp(fun, (0.0, s_end), y0, rtol=1e-8, atol=1e-10)
    phi11, phi21, phi12, phi22 = sol.y[:, -1]
    M = np.array([[phi11, phi12], [phi21, phi22]])
    return M, sol

# ---------------------------
# 5) Dispersion integration: eta'' + K(s) eta = h(s)
# ---------------------------
def integrate_dispersion(eta0, etap0, K_of_s, h_of_s, s_end):
    def fun(s, y):
        eta, etap = y
        K = K_of_s(np.array([s])) if callable(K_of_s) else K_of_s(s)
        h = h_of_s(np.array([s])) if callable(h_of_s) else h_of_s(s)
        return [etap, -K*eta + h]
    sol = solve_ivp(fun, (0.0, s_end), [eta0, etap0], rtol=1e-8, atol=1e-10)
    return sol

# ---------------------------
# Example usage (toy lattice)
# ---------------------------
if __name__=='__main__':
    # Example: simple QQD style: quad focusing, drift, quad, dip etc (you will replace with real lattice)
    elements = [
        {'type':'quad','L':0.5,'k':1.2},
        {'type':'drift','L':0.3,'k':0.0},
        {'type':'quad','L':0.5,'k':-1.0},
        {'type':'dip','L':1.0,'rho':20.0}  # dipole with bending radius -> nonzero h
    ]
    K_of_s, h_of_s, Ltot = make_piecewise_K(elements)

    # Suppose you have particle arrays u_samples, up_samples
    # (here toy)
    rng = np.random.default_rng(1)
    u = 1e-3 * rng.normal(size=10000)
    up = 1e-3 * rng.normal(size=10000)
    emit, beta0, alpha0, gamma0 = twiss_from_particles(u, up)
    print("emit, beta0, alpha0:", emit, beta0, alpha0)

    B0 = np.sqrt(beta0)
    Bp0 = -alpha0/B0

    # integrate B
    solB = integrate_B(B0, Bp0, K_of_s, Ltot)
    s_grid = solB.t
    B_vals = solB.y[0,:]
    beta_vals = B_vals**2

    # transfer matrix
    M, solM = transfer_matrix(K_of_s, Ltot)
    print("Transfer matrix M(0->L):\n", M)

    # dispersion (assume initial eta from sample or zeros)
    eta0, etap0 = 0.0, 0.0
    sol_eta = integrate_dispersion(eta0, etap0, K_of_s, h_of_s, Ltot)
    eta_end, etap_end = sol_eta.y[:, -1]
    print("eta_end, etap_end:", eta_end, etap_end)

    # offset for the largest delta in your sample:
    delta_max = 0.01
    u_offset = eta_end * delta_max
    print("max offset from dispersion:", u_offset)


### Manually input parameters to create a g4bl file from template
The g4bl files specifically take particles_before.txt as input, as specified inside the .g4bl

In [ ]:
import re
# Importing result from a DE optimizer in a specific format as below:
"""
Best result:
  B1_field             =     0.2101
  B1_width             =   122.8026
  B1_height            =   265.3664
  B1_length            =    83.8629
  Q1_gradient          =    -1.2517
  Q1_length            =   297.0612
  radius_q             =    63.9380
  Q1_z                 =   118.5929
  thickness            =   410.4403
  Q2_gradient          =    -0.1835
  Q2_width             =   174.4412
  Q2_height            =   125.0376
  Q2_length            =   183.4535
  Drift1_width         =   173.3462
  Drift1_height        =   243.1225
  Drift1_length        =   139.7656
  Drift2_width         =   191.6180
  Drift2_height        =   121.9246
  Drift2_length        =    86.8190
Final cost = 3.327e+10

"""
def parse_best_result(text):
    # Regex: captures variable name and numeric value (supports +/-, decimals, exponents)
    pattern = r"([A-Za-z0-9_]+)\s*=\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)"
    matches = re.findall(pattern, text)

    # Convert to dictionary
    result_dict = {k: float(v) for k, v in matches}

    # Optional: turn into NumPy array (just the values)
    keys = list(result_dict.keys())
    values = np.array([result_dict[k] for k in keys])

    # Print them as a Python-style array
    print(", ".join(str(v) for v in values[:-1]))

    return result_dict, keys, values
text = input("Paste the best result from optimization")
result_dict, keys, xvec = parse_best_result(text)

In [ ]:
# ---------------- USER CONFIG ----------------
G4BEAMLINE_CMD = "g4bl"
TEMPLATE_FILE = "G4_FinalCooling_dispsup_Template.g4bl"
OUTPUT_DIR = "dispsup_runs"
VD_FILENAME = "vd_dispsup.txt"   # virtual detector writes this file (ascii)
N_PARTICLES = 5000             # increase for lower noise
G4BLFILE = f"G4_FinalCooling_dispsup_Template_run.g4bl"
G4BLOUTPUT =f"/home/incik/Cooling_4D/DispSupOpts/{VD_FILENAME}"
E0 = 1e4                       # beam energy (GeV) - for a muon collider

# Let's first take in the parameters that the wedge has
import re
import math
import sys
from pathlib import Path

def read_g4bl_params(filename):
    """
    Reads 'param' definitions from a .g4bl file and returns a dict of parameter names and values.
    """
    params = {}
    pattern = re.compile(r"param\s+(?:-unset\s+)?(\w+)=([^\s#]+)")
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line.startswith("param"):
                continue
            m = pattern.search(line)
            if m:
                key, val = m.groups()
                try:
                    params[key] = float(eval(val, {"__builtins__": None, "pi": math.pi}))
                except Exception:
                    params[key] = val  # keep as string if not numeric
    return params


def compute_wedge_geometry(params):
    """
    Given parameters like absLEN3, abshgt, abswidth, abshalfangle3, compute geometry info.
    """
    W = params.get("absLEN3")
    H = params.get("abshgt")
    L = params.get("abswidth")
    half_angle_deg = params.get("abshalfangle3")
    offset = params.get("absoffset3", 0)

    if L and half_angle_deg:
        half_angle_rad = math.radians(half_angle_deg)
        L_centerline = W / math.sin(half_angle_rad)

    return {
        "Length_Wedge": L,
        "Height_Base": H,
        "Width_Base": W,
        "Half-angle": half_angle_deg,
        "Centerline_Coords": L_centerline,
        "Offset": offset
    }


if len(sys.argv) < 2:
    print("Usage: python read_wedge_params.py <file.g4bl>")
    sys.exit(1)

file_path = "G4_FinalCooling_dispsup_Template.g4bl"
params = read_g4bl_params(file_path)
geom = compute_wedge_geometry(params)

print("Parameters found:")
for k, v in params.items():
    print(f"  {k:15s} = {v}")

print("Derived geometry:")
for k, v in geom.items():
    print(f"  {k:20s}: {v}")

In [44]:
# {B1_field}, {B1_width}, {B1_height}, {B1_length}, {B1_z}, {Q1_gradient}, {Q1_length}, {radius_q}
# {B2_field}, {B2_width}, {B2_height}, {B2_length}
# {Drift1_width}, {Drift1_height}, {Drift1_length}, 
# {Drift2_width}, {Drift2_height}, {Drift2_length}

var_names = ["N_PARTICLES", "B1_field", "B1_width", "B1_height", "B1_length", 
    "Q1_gradient", "Q1_length", "radius_q", "Q1_z", "thickness",
    "Q2_gradient", "Q2_width", "Q2_height", "Q2_length",
    "Drift1_width", "Drift1_height", "Drift1_length",  
    "Drift2_width", "Drift2_height","Drift2_length", "VD_FILENAME"]
    
xvec = np.array([int(N_PARTICLES), 0.2101, 122.8026, 265.3664, 83.8629, -1.2517, 297.0612, 63.938, 118.5929, 410.4403, -0.1835, 
                174.4412, 125.0376, 183.4535, 173.3462, 243.1225, 139.7656, 191.618, 121.9246, 86.819, VD_FILENAME])

GAP = 0.1  # in mm (can be up to 1.0 safely)
A_GAP = 5 # mm
calcparams = {name: val for name, val in zip(var_names, xvec)}

L_Q1 = float(calcparams["Q1_length"])
L_D1 = float(calcparams["Drift1_length"])
L_Q2 = float(calcparams["Q2_length"])
L_D2 = float(calcparams["Drift2_length"])
L_B1 = float(calcparams["B1_length"])

wedge_end = geom["Centerline_Coords"]
Q1_z      = geom["Centerline_Coords"] + (L_Q1/2)
Drift1_z = Q1_z + (L_Q1/2) + (L_D1/2) + GAP
Q2_z     = Drift1_z + (L_D1/2) + (L_Q2/2) + GAP
Drift2_z = Q2_z + (L_Q2/2) + (L_D2/2)+ GAP
B1_z     = Drift2_z + (L_D2/2) + (L_B1/2) + GAP
VD_z     = B1_z + (L_B1/2) + 10.0 + GAP

Q1_end = Q1_z + (L_Q1/2)
D1_end = Drift1_z + (L_D1/2)
Q2_end = Q2_z + (L_Q2/2)
D2_end = Drift2_z + (L_D2/2)
B1_end = B1_z + (L_B1/2)

add_params = {"wedge_end": wedge_end, "Q1_z": Q1_z, "Drift1_z": Drift1_z, "Q2_z": Q2_z, "Drift2_z":Drift2_z, "B1_z": B1_z,"VD_z": VD_z, 
        "B1_end": B1_end, "Q2_end": Q2_end,  "D1_end": D1_end, "Q1_end": Q1_end,  "D2_end": D2_end, "N_PARTICLES": N_PARTICLES,
        "VD_FILENAME": VD_FILENAME}

calcparams.update(add_params)

for k, v in calcparams.items():
    if k == "N_PARTICLES" or k == "VD_FILENAME":
        continue
    else:
        calcparams[k] = float(v)

os.makedirs(OUTPUT_DIR, exist_ok=True)

### Creating the Run File

In [ ]:
# Write values to the prepared G4BL template
def write_input_from_template(template_path, out_path, replacements):
    with open(template_path, 'r') as f:
        txt = f.read()
    try:
        txt = txt.format(**replacements)
    except KeyError as e:
        raise RuntimeError(f"Template substitution failed; missing placeholder: {e}")
    with open(out_path, 'w') as f:
        f.write(txt)

# How to Use?
print(calcparams)
write_input_from_template(TEMPLATE_FILE, G4BLFILE, calcparams)

In [ ]:
# Run g4bl and get the Output file
"""
G4BLFILE = f"G4_FinalCooling_wdc_run_40.0_7.g4bl"
G4BLOUTPUT3 = f"vd_B1_end_achromat_40.0_7.txt"
G4BLOUTPUT2 = f"vd_wedge_end_achromat.txt"
G4BLOUTPUT1= f"vd_start_achromat.txt"
"""
if os.path.exists("field_cell.dat"):
    os.remove("field_cell.dat")
result = subprocess.run(["g4bl", G4BLFILE], capture_output=True, text=True, check=True)
print(result)

In [ ]:
# Make the results a dataframe and calculate the Courant-Snyder Parameters
df = read_trackfile(G4BLOUTPUT)
x_params, y_params, z_emit = calc_all_params(df)
print(x_params)
"""df2 = read_trackfile(G4BLOUTPUT2)
x_params1, y_params1, z_emit1 = calc_all_params(df2)
df3 = read_trackfile(G4BLOUTPUT3)
x_params3, y_params3, z_emit3 = calc_all_params(df3)"""

In [ ]:
z_emit

In [ ]:
D_dict = {"D_x": x_params[4], "D'_x": x_params[5], "D_y": y_params[4], "D'_y": y_params[5]}
print(D_dict)
cost = D_dict["D_x"]**2 + D_dict["D'_x"]**2 + D_dict["D_y"]**2 + D_dict["D'_y"]**2 
print(cost)
print(r"Epsilon_z: "+str(z_emit))

"""D_dict2 = {"D_x": x_params2[4], "D'_x": x_params2[5], "D_y": y_params2[4], "D'_y": y_params2[5]}
print(D_dict2)
cost2 = D_dict2["D_x"]**2 + D_dict2["D'_x"]**2 + D_dict2["D_y"]**2 + D_dict2["D'_y"]**2 
print(cost2)
print(r"Epsilon_z: "+str(z_emit2))

D_dict3 = {"D_x": x_params3[4], "D'_x": x_params3[5], "D_y": y_params3[4], "D'_y": y_params3[5]}
print(D_dict3)
cost3 = D_dict3["D_x"]**2 + D_dict3["D'_x"]**2 + D_dict3["D_y"]**2 + D_dict3["D'_y"]**2 
print(cost3)
print(r"Epsilon_z: "+str(z_emit3))"""

In [ ]:
with open(G4BLOUTPUT) as f:
    N_out = sum(1 for line in f if not line.startswith("#") and line.strip())
    print(N_out)

# Compute transmission (%)
trans_percent = 100.0 * N_out / int(N_PARTICLES)

print(trans_percent)

In [ ]:
def fieldEvolution():
    """ 
    Evolution of the magnetic field across elements, plotted. 
    
    """
    # Load file, skipping comment lines
    with open("field_cell.dat") as f:
        lines = f.readlines()[4:]
    
    data = np.array([[float(x) for x in line.split()] for line in lines])
    print(data)
    
    # If the file has columns: x y z Bx By Bz
    # then we can extract:
    z = data[:, 2]
    by = data[:, 4]
    
    
    # Draw shaded regions for magnets
    # Read your g4bl file
    with open("G4_FinalCooling_dispsup_Template_run.g4bl") as f:
        text = f.read()

    # Extract magnets (dipoles, quads, drifts, etc.)
    pattern = r"\s*([A-Za-z0-9_]+):\s*([-\d\.]+)"
    entries = dict(re.findall(pattern, text))
    
    # Convert numeric values
    params = {k: float(v) for k, v in entries.items()}
    print(params)
    # Collect shading regions automatically
    regions = []

    color = ["blue", "gray", "blue", "gray", "orange"]
    GAP = 1
    for idx, name in enumerate(["Q1", "Drift1", "Q2", "Drift2", "B1"]):
        if f"{name}_z" in params:
            length = params[f"{name}_length"]
            center = params[f"{name}_z"]  # sometimes z not given (assume 0)
            z1 = center - (length / 2)
            z2 = center + (length / 2)
            regions.append((z1, z2, color[idx], name))

    plt.figure(figsize=(8, 4))
    for z1, z2, color, label in regions:
        if z1 is not None and z2 is not None:
            plt.axvspan(z1, z2, color=color, alpha=0.3, label=label)
    
    plt.plot(z, by, '-o', markersize=2)
    plt.xlabel("Z (mm)")
    plt.ylabel("By (Tesla)")
    plt.title("On Axis (X0=0) Magnetic Field (By vs Z)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("Bfields.png")

fieldEvolution()

In [ ]:
def twissPlot(z_positions, emit_x, emit_y, emit_z, beta_x, beta_y, alpha_x, alpha_y, D_x, Dp_x, D_y, Dp_y, output_folder="./dispsup_plots"):
    
    # -------------------------------------------------------------------
    # CONVERT TO ARRAYS FOR PLOTTING
    # -------------------------------------------------------------------
    z_positions = np.array(z_positions)
    order = np.argsort(z_positions)

    z_positions = z_positions[order]
    emit_x = np.array(emit_x)[order]
    emit_y = np.array(emit_y)[order]
    beta_x = np.array(beta_x)[order]
    beta_y = np.array(beta_y)[order]
    alpha_x = np.array(alpha_x)[order]
    alpha_y = np.array(alpha_y)[order]
    D_x = np.array(D_x)[order]
    Dp_x = np.array(Dp_x)[order]
    D_y = np.array(D_y)[order]
    Dp_y = np.array(Dp_y)[order]
    emit_z = np.array(emit_z)[order]

    # -------------------------------------------------------------------
    # PLOTTING
    # -------------------------------------------------------------------

    def make_plot(yvals, ylabel, title, filename, labels=("x", "y")):
        plt.figure(figsize=(7,5))
        # Draw shaded regions for magnets
        # Read your g4bl file
        with open("G4_FinalCooling_dispsup_Template_run.g4bl") as f:
            text = f.read()

        # Extract magnets (dipoles, quads, drifts, etc.)
        pattern = r"\s*([A-Za-z0-9_]+):\s*([-\d\.]+)"
        entries = dict(re.findall(pattern, text))
        
        # Convert numeric values
        params = {k: float(v) for k, v in entries.items()}
        print(entries.items(), params)
        # Collect shading regions automatically
        regions = []

        last_value = 0
        
        color = ["blue", "gray", "blue", "gray", "orange"]
        for idx, name in enumerate(["Q1", "Drift1", "Q2", "Drift2", "B1"]):
            if f"{name}_z" in params:
                length = params[f"{name}_length"]
                center = params[f"{name}_z"]  # sometimes z not given (assume 0)
                z1 = center - (length / 2)
                z2 = center + (length / 2)
                regions.append((z1, z2, color[idx], name))
                                
        for z1, z2, color, label in regions:
            if z1 is not None and z2 is not None:
                plt.axvspan(z1, z2, color=color, alpha=0.3, label=label)
    
        plt.plot(z_positions, yvals[0], 'o-', label=f"{labels[0]}-plane")
        plt.plot(z_positions, yvals[1], 's--', label=f"{labels[1]}-plane")
        plt.xlabel("z [mm]")
        plt.ylabel(ylabel)
        plt.title(title)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, filename))
        plt.close()

    make_plot((emit_x, emit_y), "Emittance [mm·mrad]",
            "Emittance Evolution", "emittance_evolution.png")

    make_plot((beta_x, beta_y), "β-function [mm/rad]",
            "Beta Function Evolution", "beta_evolution.png")

    make_plot((alpha_x, alpha_y), "α-function [–]",
            "Alpha Function Evolution", "alpha_evolution.png")

    make_plot((D_x, D_y), "Dispersion D [mm]",
            "Dispersion Function", "dispersion_D.png")

    make_plot((Dp_x, Dp_y), "Dispersion Derivative D' [rad]",
            "Dispersion Derivative", "dispersion_Dprime.png")

    print(f"All plots saved in '{output_folder}'")


files = ["vd_start_dispsup.txt", "vd_wedge_end_achromat.txt", "vd_Q1_dispsup.txt", "vd_D1_dispsup.txt", "vd_Q2_dispsup.txt", "vd_D2_dispsup.txt", "vd_B1_dispsup.txt"]
z_positions = []
emit_x = []
emit_y = []
emit_z = []
beta_x = []
beta_y= []
alpha_x=[]
alpha_y=[]
D_x=[]
Dp_x=[]
D_y=[]
Dp_y=[]

for i in files:
        df1 = read_trackfile(i)
        z_positions.append(np.mean(df1["z"]))
        x_params1, y_params1, z_emit1 = calc_all_params(df1)
        ex, bx, gx, ax, Dx, Dxp = x_params1
        ey, by, gy, ay, Dy, Dyp = y_params1
        emit_x.append(ex)
        emit_y.append(ey)
        beta_x.append(bx)
        beta_y.append(by)
        alpha_x.append(ax)
        alpha_y.append(ay)
        D_x.append(Dx), Dp_x.append(Dxp), D_y.append(Dy), Dp_y.append(Dyp)
        emit_z.append(z_emit1)

twissPlot(z_positions, emit_x, emit_y, emit_z, beta_x, beta_y, alpha_x, alpha_y, D_x, Dp_x, D_y, Dp_y)